# Learning Path Recommendation System

This notebook develops and tests the personalised learning path recommendation component of the system.

The recommendation process uses:
- Academic Area
- Confidence in Key Areas
- Topics of Interest
- Academic Goal
- Predicted Performance

These inputs are combined with a structured learning-resource dataset to identify and rank relevant learning resources and generate a suggested learning pathway.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load Learning Resource Dataset

In [2]:
data_path = Path("../data/recommendation/learning_resources.csv")

resources_df = pd.read_csv(data_path)

print("Learning resource dataset loaded successfully.")
print("Dataset shape:", resources_df.shape)

resources_df.head()

Learning resource dataset loaded successfully.
Dataset shape: (70, 11)


,Resource_ID,Resource_Name,Provider,Academic_Area,Primary_Topic,Related_Topics,Supports_Skill,Difficulty_Level,Resource_Type,Description,URL
0,R001,Introduction to Marketing,University of Pennsylvania / Coursera,Business & Management,Marketing,Branding; Customer Analysis; Marketing Strategy,Business Analysis,Beginner,Course,Introduces core marketing concepts including b...,https://www.coursera.org/learn/wharton-marketing
1,R002,Introduction to Financial Accounting,University of Pennsylvania / Coursera,Business & Management,Accounting,Financial Statements; Financial Analysis; Book...,Quantitative Skills,Beginner,Course,Builds foundational skills for reading and int...,https://www.coursera.org/learn/wharton-accounting
2,R003,Introduction to Operations Management,University of Pennsylvania / Coursera,Business & Management,Operations Management,Process Analysis; Flow; Variability; Operation...,Strategic Thinking,Beginner,Course,"Introduces process analysis, operational flow,...",https://www.coursera.org/learn/wharton-operations
3,R004,Understand Human Resource Management Fundamentals,Coursera,Business & Management,Human Resource Management,Talent Management; Recruitment; Performance Ma...,Business Analysis,Beginner,Course,"Covers core HR functions, talent acquisition, ...",https://www.coursera.org/learn/understand-huma...
4,R005,Business Strategy,University of Illinois Urbana-Champaign / Cour...,Business & Management,Business Strategy,Competitive Advantage; SWOT; Business Models,Strategic Thinking,Intermediate,Course,Develops strategic analysis skills for underst...,https://www.coursera.org/learn/strategy-business


### 2.1 Initial Dataset Inspection

In [3]:
print("Columns:")
print(resources_df.columns.tolist())

print("\nData types:")
print(resources_df.dtypes)

Columns:
['Resource_ID', 'Resource_Name', 'Provider', 'Academic_Area', 'Primary_Topic', 'Related_Topics', 'Supports_Skill', 'Difficulty_Level', 'Resource_Type', 'Description', 'URL']

Data types:
Resource_ID         str
Resource_Name       str
Provider            str
Academic_Area       str
Primary_Topic       str
Related_Topics      str
Supports_Skill      str
Difficulty_Level    str
Resource_Type       str
Description         str
URL                 str
dtype: object


In [4]:
resources_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Resource_ID       70 non-null     str  
 1   Resource_Name     70 non-null     str  
 2   Provider          70 non-null     str  
 3   Academic_Area     70 non-null     str  
 4   Primary_Topic     70 non-null     str  
 5   Related_Topics    70 non-null     str  
 6   Supports_Skill    70 non-null     str  
 7   Difficulty_Level  70 non-null     str  
 8   Resource_Type     70 non-null     str  
 9   Description       70 non-null     str  
 10  URL               70 non-null     str  
dtypes: str(11)
memory usage: 29.8 KB


In [5]:
resources_df.describe(include="all").T

,count,unique,top,freq
Resource_ID,70,70,R001,1
Resource_Name,70,70,Introduction to Marketing,1
Provider,70,34,University of Colorado Boulder / Coursera,6
Academic_Area,70,10,Business & Management,7
Primary_Topic,70,51,Robotics,3
Related_Topics,70,70,Branding; Customer Analysis; Marketing Strategy,1
Supports_Skill,70,25,Critical Analysis,7
Difficulty_Level,70,3,Beginner,38
Resource_Type,70,5,Course,53
Description,70,70,Introduces core marketing concepts including b...,1


## 3. Data Quality Checks

In [6]:
print("Missing values:")
print(resources_df.isnull().sum())

print("\nTotal missing values:")
print(resources_df.isnull().sum().sum())

Missing values:
Resource_ID         0
Resource_Name       0
Provider            0
Academic_Area       0
Primary_Topic       0
Related_Topics      0
Supports_Skill      0
Difficulty_Level    0
Resource_Type       0
Description         0
URL                 0
dtype: int64

Total missing values:
0


In [7]:
print("Duplicate rows:", resources_df.duplicated().sum())

print(
    "Duplicate Resource_ID values:",
    resources_df["Resource_ID"].duplicated().sum()
)

Duplicate rows: 0
Duplicate Resource_ID values: 0


In [8]:
categorical_columns = [
    "Academic_Area",
    "Supports_Skill",
    "Difficulty_Level",
    "Resource_Type"
]

for column in categorical_columns:
    print(f"\n{column}:")
    print(resources_df[column].value_counts())


Academic_Area:
Academic_Area
Business & Management           7
Computing & IT                  7
Data Science & AI               7
Engineering & Technology        7
Psychology & Health Sciences    7
Law & Criminology               7
Media & Communications          7
Design & Creative Arts          7
Mathematics & Statistics        7
Social Sciences & Humanities    7
Name: count, dtype: int64

Supports_Skill:
Supports_Skill
Critical Analysis            7
Programming                  5
Computer Systems             4
Engineering Analysis         4
Critical Evaluation          4
Research Skills              4
Quantitative Skills          3
Statistical Analysis         3
Legal Research               3
Communication Skills         3
Media Analysis               3
Design Skills                3
Mathematical Reasoning       3
Business Analysis            2
Strategic Thinking           2
Data Analysis                2
Mathematics & Statistics     2
Technical Problem Solving    2
Research Metho

## 4. Student Profile Taxonomy

The recommendation system requires a structured relationship between the academic area selected by the student and the corresponding confidence areas and interest topics.

This taxonomy ensures that the student inputs collected by the application are consistent with the metadata used in the learning-resource dataset.

### 4.1 Skills Available by Academic Area

In [9]:
skills_by_area = (
    resources_df.groupby("Academic_Area")["Supports_Skill"]
    .unique()
    .apply(list)
)

for area, skills in skills_by_area.items():
    print(f"\n{area}")
    for skill in skills:
        print(f" - {skill}")


Business & Management
 - Business Analysis
 - Quantitative Skills
 - Strategic Thinking

Computing & IT
 - Computer Systems
 - Programming
 - Databases

Data Science & AI
 - Data Analysis
 - Programming
 - Mathematics & Statistics

Design & Creative Arts
 - Visual Communication
 - Design Skills
 - Creative Thinking

Engineering & Technology
 - Engineering Analysis
 - Technical Problem Solving
 - Mathematics

Law & Criminology
 - Critical Analysis
 - Legal Research

Mathematics & Statistics
 - Mathematical Reasoning
 - Problem Solving
 - Statistical Analysis

Media & Communications
 - Communication Skills
 - Media Analysis
 - Research Skills

Psychology & Health Sciences
 - Critical Evaluation
 - Statistical Analysis
 - Research Methods

Social Sciences & Humanities
 - Critical Analysis
 - Research Skills
 - Academic Writing


### 4.2 Topics Available by Academic Area

In [10]:
topics_by_area = (
    resources_df.groupby("Academic_Area")["Primary_Topic"]
    .unique()
    .apply(list)
)

for area, topics in topics_by_area.items():
    print(f"\n{area}")
    for topic in topics:
        print(f" - {topic}")


Business & Management
 - Marketing
 - Accounting
 - Operations Management
 - Human Resource Management
 - Business Strategy
 - Business Analytics

Computing & IT
 - Computer Systems
 - Web Development
 - Databases
 - Cybersecurity
 - Cloud Computing
 - Computer Networks
 - Software Development

Data Science & AI
 - Data Analysis
 - Artificial Intelligence
 - Statistics
 - Machine Learning
 - Deep Learning

Design & Creative Arts
 - Graphic Design
 - UX/UI Design
 - Photography
 - 3D Design

Engineering & Technology
 - Mechanical Engineering
 - Robotics
 - Renewable Energy
 - Electrical Engineering

Law & Criminology
 - Criminal Law
 - International Law
 - Human Rights
 - Business Law
 - Cybercrime

Mathematics & Statistics
 - Calculus
 - Algebra
 - Statistics
 - Probability
 - Mathematical Modelling

Media & Communications
 - Public Relations
 - Media Production
 - Advertising
 - Digital Media
 - Social Media

Psychology & Health Sciences
 - Cognitive Psychology
 - Social Psychology
 

### 4.3 Difficulty Level Distribution

Difficulty levels are examined to determine whether the resource catalogue contains sufficient progression from introductory to more advanced learning material.

In [11]:
difficulty_counts = resources_df["Difficulty_Level"].value_counts()

print("Difficulty Level Distribution:")
print(difficulty_counts)

Difficulty Level Distribution:
Difficulty_Level
Beginner        38
Intermediate    24
Advanced         8
Name: count, dtype: int64


In [12]:
difficulty_by_area = pd.crosstab(
    resources_df["Academic_Area"],
    resources_df["Difficulty_Level"]
)

difficulty_by_area

Difficulty_Level,Advanced,Beginner,Intermediate
Academic_Area,,,
Business & Management,0,5,2
Computing & IT,0,5,2
Data Science & AI,2,2,3
Design & Creative Arts,0,5,2
Engineering & Technology,2,2,3
Law & Criminology,3,0,4
Mathematics & Statistics,0,4,3
Media & Communications,0,5,2
Psychology & Health Sciences,1,6,0


## 5. Student Profile Representation

A sample student profile is defined to develop and test the recommendation logic before integration with the Streamlit application.

The profile contains the student's selected academic area, confidence ratings, topics of interest, academic goal, and predicted probability of passing. In the final application, these values will be obtained dynamically from the Academic Information and Prediction Results components.

### 5.1 Sample Student Profile

In [13]:
student_profile = {
    "Academic_Area": "Data Science & AI",
    
    "Confidence": {
        "Programming": 2,
        "Mathematics & Statistics": 4,
        "Data Analysis": 3
    },
    
    "Interests": [
        "Machine Learning",
        "Data Analysis"
    ],
    
    "Academic_Goal": "Improve academic performance",
    
    "Pass_Probability": 55.72
}

student_profile

{'Academic_Area': 'Data Science & AI',
 'Confidence': {'Programming': 2,
  'Mathematics & Statistics': 4,
  'Data Analysis': 3},
 'Interests': ['Machine Learning', 'Data Analysis'],
 'Academic_Goal': 'Improve academic performance',
 'Pass_Probability': 55.72}

### 5.2 Identify Priority Skill

The confidence ratings are used to identify the area in which the student reports the lowest confidence. This area is treated as the primary skill-development priority when generating recommendations.

In [14]:
confidence_scores = student_profile["Confidence"]

minimum_confidence = min(
    confidence_scores.values()
)

lowest_confidence_skills = [
    skill
    for skill, score in confidence_scores.items()
    if score == minimum_confidence
]

if minimum_confidence <= 3:

    priority_skills = lowest_confidence_skills

    if len(priority_skills) == 1:
        priority_skill = priority_skills[0]
    else:
        priority_skill = ", ".join(priority_skills)

else:

    priority_skills = []

    priority_skill = (
        "No major low-confidence area identified"
    )


print("Priority skill:", priority_skill)
print("Priority skills:", priority_skills)
print("Minimum confidence:", minimum_confidence)

Priority skill: Programming
Priority skills: ['Programming']
Minimum confidence: 2


### 5.3 Academic Area Filtering

Academic area is used as an initial hard filter. This ensures that students are only considered for resources relevant to their selected field of study before personalised ranking is performed.

In [15]:
selected_area = student_profile["Academic_Area"]

candidate_resources = resources_df[
    resources_df["Academic_Area"] == selected_area
].copy()

print("Selected academic area:", selected_area)
print("Candidate resources:", len(candidate_resources))

candidate_resources[
    [
        "Resource_Name",
        "Primary_Topic",
        "Supports_Skill",
        "Difficulty_Level"
    ]
]

Selected academic area: Data Science & AI
Candidate resources: 7


,Resource_Name,Primary_Topic,Supports_Skill,Difficulty_Level
14,What is Data Science?,Data Analysis,Data Analysis,Beginner
15,Python for Everybody,Artificial Intelligence,Programming,Beginner
16,Statistics for Data Science with Python,Statistics,Mathematics & Statistics,Intermediate
17,Data Analysis with Python,Data Analysis,Data Analysis,Intermediate
18,Machine Learning Specialization,Machine Learning,Programming,Intermediate
19,Deep Learning Specialization,Deep Learning,Programming,Advanced
20,Data Science: Statistics and Machine Learning,Machine Learning,Mathematics & Statistics,Advanced


### 5.4 Priority Skill Matching

Candidate resources are evaluated according to whether they support the student's lowest-confidence area. Resources that directly address this area will later receive a higher recommendation score.

In [16]:
candidate_resources["Priority_Skill_Match"] = (
    candidate_resources["Supports_Skill"]
    .isin(priority_skills)
    .astype(int)
)

candidate_resources[
    [
        "Resource_Name",
        "Supports_Skill",
        "Priority_Skill_Match"
    ]
]

,Resource_Name,Supports_Skill,Priority_Skill_Match
14,What is Data Science?,Data Analysis,0
15,Python for Everybody,Programming,1
16,Statistics for Data Science with Python,Mathematics & Statistics,0
17,Data Analysis with Python,Data Analysis,0
18,Machine Learning Specialization,Programming,1
19,Deep Learning Specialization,Programming,1
20,Data Science: Statistics and Machine Learning,Mathematics & Statistics,0


### 5.5 Interest Matching

The student's selected topics of interest are compared with the topic metadata associated with each candidate resource. Both the primary topic and related topics are considered so that resources can match a student's interests even when the wording is not restricted to a single topic field.

In [17]:
student_interests = [
    interest.lower()
    for interest in student_profile["Interests"]
]

def calculate_interest_matches(row):
    
    resource_text = (
        str(row["Primary_Topic"]) + " " +
        str(row["Related_Topics"])
    ).lower()
    
    matches = sum(
        interest in resource_text
        for interest in student_interests
    )
    
    return matches


candidate_resources["Interest_Matches"] = (
    candidate_resources.apply(
        calculate_interest_matches,
        axis=1
    )
)

candidate_resources[
    [
        "Resource_Name",
        "Primary_Topic",
        "Related_Topics",
        "Interest_Matches"
    ]
]

,Resource_Name,Primary_Topic,Related_Topics,Interest_Matches
14,What is Data Science?,Data Analysis,Data Science; Analytics; Data Roles; Data Meth...,1
15,Python for Everybody,Artificial Intelligence,Python; Programming Fundamentals; Data Processing,0
16,Statistics for Data Science with Python,Statistics,Descriptive Statistics; Hypothesis Testing; Re...,0
17,Data Analysis with Python,Data Analysis,Pandas; NumPy; Data Cleaning; EDA; Regression,1
18,Machine Learning Specialization,Machine Learning,Supervised Learning; Unsupervised Learning; Ne...,1
19,Deep Learning Specialization,Deep Learning,Neural Networks; CNNs; Sequence Models; Optimi...,0
20,Data Science: Statistics and Machine Learning,Machine Learning,Statistical Inference; Regression; Prediction;...,1


## 6. Recommendation Scoring and Ranking

Candidate resources are ranked using multiple components of the student profile. The scoring mechanism combines the student's confidence-based skill-development needs, stated topic interests, resource difficulty, and textual similarity between the student profile and resource metadata.

This hybrid approach allows recommendations to consider both explicit decision rules and similarity-based matching rather than relying on a single criterion.

### 6.1 Confidence-Based Skill Score

A confidence-based score is calculated for each resource according to the student's self-reported confidence in the skill supported by that resource. Lower confidence produces a higher development-need score, allowing the recommender to prioritise resources addressing areas in which the student requires greater support.

In [18]:
def calculate_skill_need(supported_skill):
    
    confidence = student_profile["Confidence"].get(
        supported_skill,
        3
    )
    
    # Convert 1–5 confidence into 0–1 development need
    skill_need = (5 - confidence) / 4
    
    return skill_need


candidate_resources["Skill_Need_Score"] = (
    candidate_resources["Supports_Skill"]
    .apply(calculate_skill_need)
)

candidate_resources[
    [
        "Resource_Name",
        "Supports_Skill",
        "Skill_Need_Score"
    ]
]

,Resource_Name,Supports_Skill,Skill_Need_Score
14,What is Data Science?,Data Analysis,0.50
15,Python for Everybody,Programming,0.75
16,Statistics for Data Science with Python,Mathematics & Statistics,0.25
17,Data Analysis with Python,Data Analysis,0.50
18,Machine Learning Specialization,Programming,0.75
19,Deep Learning Specialization,Programming,0.75
20,Data Science: Statistics and Machine Learning,Mathematics & Statistics,0.25


### 6.2 Interest Relevance Score

The number of matched interests is normalised according to the total number of interests selected by the student. This produces a consistent score between 0 and 1 representing the degree to which each resource corresponds with the student's stated interests.

In [19]:
number_of_interests = len(student_profile["Interests"])

candidate_resources["Interest_Score"] = (
    candidate_resources["Interest_Matches"]
    / number_of_interests
)

candidate_resources[
    [
        "Resource_Name",
        "Interest_Matches",
        "Interest_Score"
    ]
]

,Resource_Name,Interest_Matches,Interest_Score
14,What is Data Science?,1,0.5
15,Python for Everybody,0,0.0
16,Statistics for Data Science with Python,0,0.0
17,Data Analysis with Python,1,0.5
18,Machine Learning Specialization,1,0.5
19,Deep Learning Specialization,0,0.0
20,Data Science: Statistics and Machine Learning,1,0.5


### 6.3 Difficulty Appropriateness

The predicted probability of passing is used as a supporting signal when determining suitable resource difficulty. Students with lower predicted probabilities are initially prioritised towards foundational resources, while students with stronger predicted performance can progressively receive more advanced material.

The prediction does not determine recommendations independently; it is used alongside confidence levels and interests.

In [20]:
pass_probability = student_profile["Pass_Probability"]

if pass_probability < 50:
    preferred_difficulties = ["Beginner"]

elif pass_probability <= 75:
    preferred_difficulties = ["Beginner", "Intermediate"]

else:
    preferred_difficulties = ["Intermediate", "Advanced"]


candidate_resources["Difficulty_Score"] = (
    candidate_resources["Difficulty_Level"]
    .apply(
        lambda x: 1.0 if x in preferred_difficulties else 0.0
    )
)

print("Pass probability:", pass_probability)
print("Preferred difficulties:", preferred_difficulties)

candidate_resources[
    [
        "Resource_Name",
        "Difficulty_Level",
        "Difficulty_Score"
    ]
]

Pass probability: 55.72
Preferred difficulties: ['Beginner', 'Intermediate']


,Resource_Name,Difficulty_Level,Difficulty_Score
14,What is Data Science?,Beginner,1.0
15,Python for Everybody,Beginner,1.0
16,Statistics for Data Science with Python,Intermediate,1.0
17,Data Analysis with Python,Intermediate,1.0
18,Machine Learning Specialization,Intermediate,1.0
19,Deep Learning Specialization,Advanced,0.0
20,Data Science: Statistics and Machine Learning,Advanced,0.0


### 6.4 Similarity-Based Matching

A text-based representation is created for each candidate learning resource using its academic area, topic, related topics, supported skill, difficulty level and description.

The student's academic profile is represented in a similar format. TF-IDF is used to transform both the student profile and resource information into numerical vectors, after which cosine similarity is calculated to measure how closely each learning resource matches the student's profile.

In [21]:
candidate_resources["Combined_Text"] = (
    candidate_resources["Academic_Area"].astype(str) + " " +
    candidate_resources["Primary_Topic"].astype(str) + " " +
    candidate_resources["Related_Topics"].astype(str) + " " +
    candidate_resources["Supports_Skill"].astype(str) + " " +
    candidate_resources["Difficulty_Level"].astype(str) + " " +
    candidate_resources["Description"].astype(str)
)

candidate_resources[
    [
        "Resource_Name",
        "Combined_Text"
    ]
]

,Resource_Name,Combined_Text
14,What is Data Science?,Data Science & AI Data Analysis Data Science; ...
15,Python for Everybody,Data Science & AI Artificial Intelligence Pyth...
16,Statistics for Data Science with Python,Data Science & AI Statistics Descriptive Stati...
17,Data Analysis with Python,Data Science & AI Data Analysis Pandas; NumPy;...
18,Machine Learning Specialization,Data Science & AI Machine Learning Supervised ...
19,Deep Learning Specialization,Data Science & AI Deep Learning Neural Network...
20,Data Science: Statistics and Machine Learning,Data Science & AI Machine Learning Statistical...


### 6.5 Student Similarity Profile

In [22]:
student_profile_components = [
    student_profile["Academic_Area"]
]

if priority_skills:
    student_profile_components.append(
        " ".join(priority_skills)
    )

if student_profile["Interests"]:
    student_profile_components.append(
        " ".join(student_profile["Interests"])
    )

if student_profile["Academic_Goal"]:
    student_profile_components.append(
        student_profile["Academic_Goal"]
    )

student_text = " ".join(
    student_profile_components
)

print("Student similarity profile:")
print(student_text)

Student similarity profile:
Data Science & AI Programming Machine Learning Data Analysis Improve academic performance


### 6.6 Cosine Similarity Calculation

TF-IDF vectors are generated for the student profile and candidate resources. Cosine similarity is then used to calculate a similarity score between the student profile and each resource.

In [23]:
documents = [student_text] + candidate_resources["Combined_Text"].tolist()

tfidf = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(documents)

student_vector = tfidf_matrix[0]
resource_vectors = tfidf_matrix[1:]

similarity_scores = cosine_similarity(
    student_vector,
    resource_vectors
)[0]

candidate_resources["Similarity_Score"] = similarity_scores

In [24]:
candidate_resources[
    [
        "Resource_Name",
        "Primary_Topic",
        "Supports_Skill",
        "Similarity_Score"
    ]
].sort_values(
    by="Similarity_Score",
    ascending=False
).round(4)

,Resource_Name,Primary_Topic,Supports_Skill,Similarity_Score
15,Python for Everybody,Artificial Intelligence,Programming,0.3607
14,What is Data Science?,Data Analysis,Data Analysis,0.3054
18,Machine Learning Specialization,Machine Learning,Programming,0.2927
17,Data Analysis with Python,Data Analysis,Data Analysis,0.2795
20,Data Science: Statistics and Machine Learning,Machine Learning,Mathematics & Statistics,0.2606
19,Deep Learning Specialization,Deep Learning,Programming,0.1406
16,Statistics for Data Science with Python,Statistics,Mathematics & Statistics,0.0868


## 7. Final Recommendation Score and Ranking

The individual recommendation components are combined into a weighted score to rank the candidate learning resources.

The final score considers:
- confidence-based skill-development need,
- topic-interest relevance,
- similarity between the student profile and resource metadata,
- and difficulty appropriateness informed by predicted performance.

Greater weight is given to skill-development need because the recommendation system is intended to prioritise areas in which the student reports lower confidence. Interest relevance and similarity-based matching are also given substantial importance, while predicted performance is used as a smaller supporting signal for resource difficulty.

In [25]:
SKILL_WEIGHT = 0.35
INTEREST_WEIGHT = 0.25
SIMILARITY_WEIGHT = 0.25
DIFFICULTY_WEIGHT = 0.15

candidate_resources["Recommendation_Score"] = (
    candidate_resources["Skill_Need_Score"] * SKILL_WEIGHT
    + candidate_resources["Interest_Score"] * INTEREST_WEIGHT
    + candidate_resources["Similarity_Score"] * SIMILARITY_WEIGHT
    + candidate_resources["Difficulty_Score"] * DIFFICULTY_WEIGHT
)

In [26]:
ranked_resources = candidate_resources.sort_values(
    by="Recommendation_Score",
    ascending=False
).copy()

ranked_resources[
    [
        "Resource_Name",
        "Supports_Skill",
        "Primary_Topic",
        "Difficulty_Level",
        "Skill_Need_Score",
        "Interest_Score",
        "Similarity_Score",
        "Difficulty_Score",
        "Recommendation_Score"
    ]
].round(4)

,Resource_Name,Supports_Skill,Primary_Topic,Difficulty_Level,Skill_Need_Score,Interest_Score,Similarity_Score,Difficulty_Score,Recommendation_Score
18,Machine Learning Specialization,Programming,Machine Learning,Intermediate,0.75,0.5,0.2927,1.0,0.6107
14,What is Data Science?,Data Analysis,Data Analysis,Beginner,0.50,0.5,0.3054,1.0,0.5264
17,Data Analysis with Python,Data Analysis,Data Analysis,Intermediate,0.50,0.5,0.2795,1.0,0.5199
15,Python for Everybody,Programming,Artificial Intelligence,Beginner,0.75,0.0,0.3607,1.0,0.5027
19,Deep Learning Specialization,Programming,Deep Learning,Advanced,0.75,0.0,0.1406,0.0,0.2977
20,Data Science: Statistics and Machine Learning,Mathematics & Statistics,Machine Learning,Advanced,0.25,0.5,0.2606,0.0,0.2776
16,Statistics for Data Science with Python,Mathematics & Statistics,Statistics,Intermediate,0.25,0.0,0.0868,1.0,0.2592


### 7.1 Recommendation Match Score

The weighted recommendation score is converted to a percentage for presentation within the user interface. This percentage represents the relative match between the student's profile and each learning resource according to the defined recommendation criteria.

In [27]:
ranked_resources["Match_Score"] = (
    ranked_resources["Recommendation_Score"] * 100
).round(1)

ranked_resources[
    [
        "Resource_Name",
        "Supports_Skill",
        "Difficulty_Level",
        "Match_Score"
    ]
]

,Resource_Name,Supports_Skill,Difficulty_Level,Match_Score
18,Machine Learning Specialization,Programming,Intermediate,61.1
14,What is Data Science?,Data Analysis,Beginner,52.6
17,Data Analysis with Python,Data Analysis,Intermediate,52.0
15,Python for Everybody,Programming,Beginner,50.3
19,Deep Learning Specialization,Programming,Advanced,29.8
20,Data Science: Statistics and Machine Learning,Mathematics & Statistics,Advanced,27.8
16,Statistics for Data Science with Python,Mathematics & Statistics,Intermediate,25.9


### 7.2 Top Recommended Resources

The five highest-ranked resources are selected for presentation within the recommendation interface.

In [28]:
top_recommendations = ranked_resources.head(5).copy()

top_recommendations[
    [
        "Resource_Name",
        "Provider",
        "Primary_Topic",
        "Supports_Skill",
        "Difficulty_Level",
        "Resource_Type",
        "Match_Score"
    ]
]

,Resource_Name,Provider,Primary_Topic,Supports_Skill,Difficulty_Level,Resource_Type,Match_Score
18,Machine Learning Specialization,DeepLearning.AI & Stanford Online / Coursera,Machine Learning,Programming,Intermediate,Specialization,61.1
14,What is Data Science?,IBM / Coursera,Data Analysis,Data Analysis,Beginner,Course,52.6
17,Data Analysis with Python,IBM / Coursera,Data Analysis,Data Analysis,Intermediate,Course,52.0
15,Python for Everybody,University of Michigan / Coursera,Artificial Intelligence,Programming,Beginner,Specialization,50.3
19,Deep Learning Specialization,DeepLearning.AI / Coursera,Deep Learning,Programming,Advanced,Specialization,29.8


### 7.3 Recommendation Explanation

A short explanation is generated for each recommendation to show how the resource relates to the student's confidence levels, interests, and learning needs.

In [29]:
def generate_recommendation_reason(row):
    reasons = []

    supported_skill = row["Supports_Skill"]
    confidence = student_profile["Confidence"].get(supported_skill, 3)

    if confidence <= 2:
        reasons.append(
            f"targets your lower-confidence area in {supported_skill}"
        )
    elif confidence == 3:
        reasons.append(
            f"supports further development in {supported_skill}"
        )

    if row["Interest_Score"] > 0:
        reasons.append(
            "matches one or more of your selected interests"
        )

    if row["Difficulty_Score"] == 1.0:
        reasons.append(
            f"provides an appropriate {row['Difficulty_Level'].lower()} level"
        )

    if not reasons:
        reasons.append(
            "is relevant to your selected academic area and learning profile"
        )

    return "; ".join(reasons)


top_recommendations["Recommendation_Reason"] = (
    top_recommendations.apply(
        generate_recommendation_reason,
        axis=1
    )
)

top_recommendations[
    [
        "Resource_Name",
        "Match_Score",
        "Recommendation_Reason"
    ]
]

,Resource_Name,Match_Score,Recommendation_Reason
18,Machine Learning Specialization,61.1,targets your lower-confidence area in Programm...
14,What is Data Science?,52.6,supports further development in Data Analysis;...
17,Data Analysis with Python,52.0,supports further development in Data Analysis;...
15,Python for Everybody,50.3,targets your lower-confidence area in Programm...
19,Deep Learning Specialization,29.8,targets your lower-confidence area in Programming


## 8. Suggested Learning Path Generation

In addition to ranking individual learning resources, the system generates a structured learning path to provide students with a logical progression through suitable learning materials.

Recommended resources are organised according to their difficulty level:

- Beginner resources form the Foundations stage.
- Intermediate resources form the Core Skills stage.
- Advanced resources form the Advanced Topics stage.

Within each stage, resources are ordered according to their personalised recommendation score. This allows the learning path to combine progressive difficulty with the student's confidence levels, interests, academic area and predicted performance.

In [30]:
difficulty_to_stage = {
    "Beginner": "1. Foundations",
    "Intermediate": "2. Core Skills",
    "Advanced": "3. Advanced Topics"
}

ranked_resources["Learning_Path_Stage"] = (
    ranked_resources["Difficulty_Level"]
    .map(difficulty_to_stage)
)

learning_path = ranked_resources.sort_values(
    by=["Learning_Path_Stage", "Recommendation_Score"],
    ascending=[True, False]
).copy()

learning_path[
    [
        "Learning_Path_Stage",
        "Resource_Name",
        "Supports_Skill",
        "Primary_Topic",
        "Difficulty_Level",
        "Match_Score"
    ]
]

,Learning_Path_Stage,Resource_Name,Supports_Skill,Primary_Topic,Difficulty_Level,Match_Score
14,1. Foundations,What is Data Science?,Data Analysis,Data Analysis,Beginner,52.6
15,1. Foundations,Python for Everybody,Programming,Artificial Intelligence,Beginner,50.3
18,2. Core Skills,Machine Learning Specialization,Programming,Machine Learning,Intermediate,61.1
17,2. Core Skills,Data Analysis with Python,Data Analysis,Data Analysis,Intermediate,52.0
16,2. Core Skills,Statistics for Data Science with Python,Mathematics & Statistics,Statistics,Intermediate,25.9
19,3. Advanced Topics,Deep Learning Specialization,Programming,Deep Learning,Advanced,29.8
20,3. Advanced Topics,Data Science: Statistics and Machine Learning,Mathematics & Statistics,Machine Learning,Advanced,27.8


### 8.1 Learning Path Stage Selection

A limited number of highly ranked resources are selected from each difficulty stage to produce a concise and interpretable learning pathway for presentation within the prototype.

In [31]:
pathway_resources = (
    learning_path[
        learning_path["Difficulty_Level"].isin(
            preferred_difficulties
        )
    ]
    .groupby(
        "Learning_Path_Stage",
        group_keys=False
    )
    .head(2)
    .copy()
)

pathway_resources[
    [
        "Learning_Path_Stage",
        "Resource_Name",
        "Supports_Skill",
        "Difficulty_Level",
        "Match_Score"
    ]
]

,Learning_Path_Stage,Resource_Name,Supports_Skill,Difficulty_Level,Match_Score
14,1. Foundations,What is Data Science?,Data Analysis,Beginner,52.6
15,1. Foundations,Python for Everybody,Programming,Beginner,50.3
18,2. Core Skills,Machine Learning Specialization,Programming,Intermediate,61.1
17,2. Core Skills,Data Analysis with Python,Data Analysis,Intermediate,52.0


## 9. Recommendation System Testing

The recommendation system is tested using multiple student scenarios representing different academic areas, confidence profiles, interests and predicted performance levels.

The purpose of this testing is to examine whether the recommendation process:

- filters resources to the correct academic area,
- responds to differences in confidence levels,
- reflects the student's selected interests,
- adjusts resource difficulty appropriately according to predicted performance,
- and generates a logical learning pathway.

### 9.1 Reusable Recommendation Function

In [32]:
def recommend_resources(
    academic_area,
    confidence_scores,
    interests,
    academic_goal,
    pass_probability,
    top_n=5
):

    # -----------------------------
    # 1. Filter by academic area
    # -----------------------------

    candidates = resources_df[
        resources_df["Academic_Area"] == academic_area
    ].copy()


    # -----------------------------
    # 2. Identify priority skill(s)
    # -----------------------------

    minimum_confidence = min(
        confidence_scores.values()
    )

    lowest_confidence_skills = [
        skill
        for skill, score in confidence_scores.items()
        if score == minimum_confidence
    ]

    if minimum_confidence <= 3:

        priority_skills = lowest_confidence_skills

        if len(priority_skills) == 1:
            priority_skill = priority_skills[0]

        else:
            priority_skill = ", ".join(
                priority_skills
            )

    else:

        priority_skills = []

        priority_skill = (
            "No major low-confidence area identified"
        )


    # -----------------------------
    # 3. Confidence-based skill need
    # -----------------------------

    def calculate_skill_need(supported_skill):

        confidence = confidence_scores.get(
            supported_skill,
            3
        )

        return (5 - confidence) / 4


    candidates["Skill_Need_Score"] = (
        candidates["Supports_Skill"]
        .apply(calculate_skill_need)
    )


    # -----------------------------
    # 4. Interest relevance
    # -----------------------------

    interests_lower = [
        interest.lower()
        for interest in interests
    ]


    def calculate_interest_matches(row):

        resource_text = (
            str(row["Primary_Topic"])
            + " "
            + str(row["Related_Topics"])
        ).lower()

        return sum(
            interest in resource_text
            for interest in interests_lower
        )


    candidates["Interest_Matches"] = (
        candidates.apply(
            calculate_interest_matches,
            axis=1
        )
    )


    if len(interests) > 0:

        candidates["Interest_Score"] = (
            candidates["Interest_Matches"]
            / len(interests)
        )

    else:

        candidates["Interest_Score"] = 0.0


    # -----------------------------
    # 5. Difficulty appropriateness
    # -----------------------------

    if pass_probability < 50:

        preferred_difficulties = [
            "Beginner"
        ]

    elif pass_probability <= 75:

        preferred_difficulties = [
            "Beginner",
            "Intermediate"
        ]

    else:

        preferred_difficulties = [
            "Intermediate",
            "Advanced"
        ]


    candidates["Difficulty_Score"] = (
        candidates["Difficulty_Level"]
        .apply(
            lambda x:
            1.0 if x in preferred_difficulties
            else 0.0
        )
    )


    # -----------------------------
    # 6. Similarity-based matching
    # -----------------------------

    candidates["Combined_Text"] = (
        candidates["Academic_Area"].astype(str)
        + " "
        + candidates["Primary_Topic"].astype(str)
        + " "
        + candidates["Related_Topics"].astype(str)
        + " "
        + candidates["Supports_Skill"].astype(str)
        + " "
        + candidates["Difficulty_Level"].astype(str)
        + " "
        + candidates["Description"].astype(str)
    )


    # Build student profile text

    student_profile_components = [
        academic_area
    ]

    if priority_skills:
        student_profile_components.append(
            " ".join(priority_skills)
        )

    if interests:
        student_profile_components.append(
            " ".join(interests)
        )

    if academic_goal:
        student_profile_components.append(
            academic_goal
        )


    student_text = " ".join(
        student_profile_components
    )


    documents = (
        [student_text]
        + candidates["Combined_Text"].tolist()
    )


    tfidf = TfidfVectorizer(
        stop_words="english"
    )


    tfidf_matrix = tfidf.fit_transform(
        documents
    )


    similarity_scores = cosine_similarity(
        tfidf_matrix[0],
        tfidf_matrix[1:]
    )[0]


    candidates["Similarity_Score"] = (
        similarity_scores
    )


    # -----------------------------
    # 7. Final weighted score
    # -----------------------------

    SKILL_WEIGHT = 0.35
    INTEREST_WEIGHT = 0.25
    SIMILARITY_WEIGHT = 0.25
    DIFFICULTY_WEIGHT = 0.15


    candidates["Recommendation_Score"] = (
        candidates["Skill_Need_Score"]
        * SKILL_WEIGHT

        + candidates["Interest_Score"]
        * INTEREST_WEIGHT

        + candidates["Similarity_Score"]
        * SIMILARITY_WEIGHT

        + candidates["Difficulty_Score"]
        * DIFFICULTY_WEIGHT
    )


    candidates["Match_Score"] = (
        candidates["Recommendation_Score"]
        * 100
    ).round(1)


    # -----------------------------
    # 8. Rank resources
    # -----------------------------

    ranked = candidates.sort_values(
        by="Recommendation_Score",
        ascending=False
    ).copy()


    # -----------------------------
    # 9. Learning path stages
    # -----------------------------

    difficulty_to_stage = {
        "Beginner": "1. Foundations",
        "Intermediate": "2. Core Skills",
        "Advanced": "3. Advanced Topics"
    }


    ranked["Learning_Path_Stage"] = (
        ranked["Difficulty_Level"]
        .map(difficulty_to_stage)
    )


    # Only include preferred difficulty levels
    # in the suggested learning path

    pathway = ranked[
        ranked["Difficulty_Level"].isin(
            preferred_difficulties
        )
    ].copy()


    pathway = pathway.sort_values(
        by=[
            "Learning_Path_Stage",
            "Recommendation_Score"
        ],
        ascending=[
            True,
            False
        ]
    ).copy()


    pathway = (
        pathway
        .groupby(
            "Learning_Path_Stage",
            group_keys=False
        )
        .head(2)
        .copy()
    )


    # -----------------------------
    # 10. Return results
    # -----------------------------

    return {
        "priority_skill": priority_skill,
        "priority_skills": priority_skills,
        "preferred_difficulties": preferred_difficulties,
        "top_recommendations": ranked.head(top_n),
        "learning_path": pathway
    }

In [33]:
scenario_1 = recommend_resources(
    academic_area="Data Science & AI",
    
    confidence_scores={
        "Programming": 1,
        "Mathematics & Statistics": 2,
        "Data Analysis": 2
    },
    
    interests=[
        "Machine Learning",
        "Data Analysis"
    ],
    
    academic_goal="Improve academic performance",
    
    pass_probability=35.0
)

print("Priority skill:")
print(scenario_1["priority_skill"])

print("\nPreferred difficulties:")
print(scenario_1["preferred_difficulties"])

scenario_1["top_recommendations"][
    [
        "Resource_Name",
        "Supports_Skill",
        "Difficulty_Level",
        "Match_Score"
    ]
]

Priority skill:


Programming

Preferred difficulties:
['Beginner']


,Resource_Name,Supports_Skill,Difficulty_Level,Match_Score
14,What is Data Science?,Data Analysis,Beginner,61.4
15,Python for Everybody,Programming,Beginner,59.0
18,Machine Learning Specialization,Programming,Intermediate,54.8
17,Data Analysis with Python,Data Analysis,Intermediate,45.7
20,Data Science: Statistics and Machine Learning,Mathematics & Statistics,Advanced,45.3


In [34]:
scenario_1["learning_path"][
    [
        "Learning_Path_Stage",
        "Resource_Name",
        "Difficulty_Level",
        "Match_Score"
    ]
]

,Learning_Path_Stage,Resource_Name,Difficulty_Level,Match_Score
14,1. Foundations,What is Data Science?,Beginner,61.4
15,1. Foundations,Python for Everybody,Beginner,59.0


In [35]:
scenario_2 = recommend_resources(
    academic_area="Data Science & AI",
    
    confidence_scores={
        "Programming": 4,
        "Mathematics & Statistics": 4,
        "Data Analysis": 5
    },
    
    interests=[
        "Machine Learning",
        "Deep Learning"
    ],
    
    academic_goal="Develop advanced technical skills",
    
    pass_probability=88.0
)

print("Priority skill:")
print(scenario_2["priority_skill"])

print("\nPreferred difficulties:")
print(scenario_2["preferred_difficulties"])

scenario_2["top_recommendations"][
    [
        "Resource_Name",
        "Supports_Skill",
        "Difficulty_Level",
        "Match_Score"
    ]
]

Priority skill:
No major low-confidence area identified

Preferred difficulties:
['Intermediate', 'Advanced']


,Resource_Name,Supports_Skill,Difficulty_Level,Match_Score
18,Machine Learning Specialization,Programming,Intermediate,44.7
20,Data Science: Statistics and Machine Learning,Mathematics & Statistics,Advanced,44.0
19,Deep Learning Specialization,Programming,Advanced,42.2
16,Statistics for Data Science with Python,Mathematics & Statistics,Intermediate,26.3
17,Data Analysis with Python,Data Analysis,Intermediate,17.3


In [36]:
scenario_3 = recommend_resources(
    academic_area="Computing & IT",
    
    confidence_scores={
        "Programming": 2,
        "Databases": 4,
        "Computer Systems": 3
    },
    
    interests=[
        "Cybersecurity",
        "Cloud Computing"
    ],
    
    academic_goal="Improve practical IT skills",
    
    pass_probability=62.0
)

print("Priority skill:")
print(scenario_3["priority_skill"])

scenario_3["top_recommendations"][
    [
        "Resource_Name",
        "Primary_Topic",
        "Supports_Skill",
        "Difficulty_Level",
        "Match_Score"
    ]
]

Priority skill:
Programming


,Resource_Name,Primary_Topic,Supports_Skill,Difficulty_Level,Match_Score
11,AWS Cloud Technical Essentials,Cloud Computing,Computer Systems,Intermediate,52.1
7,Information Technology (IT) Fundamentals for E...,Computer Systems,Computer Systems,Beginner,49.4
10,Cybersecurity for Everyone,Cybersecurity,Computer Systems,Beginner,48.4
8,Web Design for Everybody,Web Development,Programming,Beginner,45.8
13,"Introduction to Software, Programming, and Dat...",Software Development,Programming,Beginner,45.4


In [37]:
scenario_4 = recommend_resources(
    academic_area="Business & Management",
    
    confidence_scores={
        "Business Analysis": 3,
        "Quantitative Skills": 2,
        "Strategic Thinking": 4
    },
    
    interests=[
        "Finance",
        "Business Strategy"
    ],
    
    academic_goal="Improve business decision-making skills",
    
    pass_probability=58.0
)

scenario_4["top_recommendations"][
    [
        "Resource_Name",
        "Primary_Topic",
        "Supports_Skill",
        "Difficulty_Level",
        "Match_Score"
    ]
]

,Resource_Name,Primary_Topic,Supports_Skill,Difficulty_Level,Match_Score
6,Business Analytics for Decision Making,Business Analytics,Quantitative Skills,Intermediate,51.4
5,Introduction to Business Analytics,Business Analytics,Quantitative Skills,Beginner,49.7
1,Introduction to Financial Accounting,Accounting,Quantitative Skills,Beginner,46.0
4,Business Strategy,Business Strategy,Strategic Thinking,Intermediate,43.4
0,Introduction to Marketing,Marketing,Business Analysis,Beginner,36.2


In [38]:
scenario_5 = recommend_resources(
    academic_area="Psychology & Health Sciences",
    
    confidence_scores={
        "Research Methods": 2,
        "Statistical Analysis": 3,
        "Critical Evaluation": 4
    },
    
    interests=[
        "Mental Health",
        "Psychological Research"
    ],
    
    academic_goal="Strengthen research and academic skills",
    
    pass_probability=70.0
)

scenario_5["top_recommendations"][
    [
        "Resource_Name",
        "Primary_Topic",
        "Supports_Skill",
        "Difficulty_Level",
        "Match_Score"
    ]
]

,Resource_Name,Primary_Topic,Supports_Skill,Difficulty_Level,Match_Score
33,Understanding Research Methods,Psychological Research,Research Methods,Beginner,70.9
34,The Science of Well-Being,Health Psychology,Research Methods,Beginner,49.9
30,Positive Psychiatry and Mental Health,Mental Health,Critical Evaluation,Beginner,41.4
28,Introduction to Psychology,Cognitive Psychology,Critical Evaluation,Beginner,40.7
32,Psychological First Aid,Mental Health,Critical Evaluation,Beginner,39.4


### 9.7 Recommendation Testing Summary

Scenario-based testing was conducted using student profiles with different academic areas, confidence levels, interests and predicted performance probabilities.

The tests demonstrated that the recommendation system responds to changes in student information in several ways:

- academic-area filtering prevented resources from unrelated disciplines from being recommended;
- lower confidence in a skill increased the priority of resources supporting that skill;
- selected interests influenced the relevance and ranking of learning resources;
- predicted performance influenced the preferred difficulty of recommended resources;
- similarity-based matching differentiated between resources according to the student's profile;
- and the learning-path component organised relevant resources into progressive Foundations, Core Skills and Advanced Topics stages where suitable resources were available.

The testing therefore indicates that the recommendation approach can generate different personalised resource rankings and learning pathways for different student profiles.

In [39]:
testing_summary = pd.DataFrame({
    "Scenario": [
        "Data Science & AI - Lower Performance",
        "Data Science & AI - Stronger Performance",
        "Computing & IT",
        "Business & Management",
        "Psychology & Health Sciences"
    ],
    "Priority_Skill": [
        scenario_1["priority_skill"],
        scenario_2["priority_skill"],
        scenario_3["priority_skill"],
        scenario_4["priority_skill"],
        scenario_5["priority_skill"]
    ],
    "Preferred_Difficulty": [
        ", ".join(scenario_1["preferred_difficulties"]),
        ", ".join(scenario_2["preferred_difficulties"]),
        ", ".join(scenario_3["preferred_difficulties"]),
        ", ".join(scenario_4["preferred_difficulties"]),
        ", ".join(scenario_5["preferred_difficulties"])
    ],
    "Top_Recommendation": [
        scenario_1["top_recommendations"].iloc[0]["Resource_Name"],
        scenario_2["top_recommendations"].iloc[0]["Resource_Name"],
        scenario_3["top_recommendations"].iloc[0]["Resource_Name"],
        scenario_4["top_recommendations"].iloc[0]["Resource_Name"],
        scenario_5["top_recommendations"].iloc[0]["Resource_Name"]
    ]
})

testing_summary

,Scenario,Priority_Skill,Preferred_Difficulty,Top_Recommendation
0,Data Science & AI - Lower Performance,Programming,Beginner,What is Data Science?
1,Data Science & AI - Stronger Performance,No major low-confidence area identified,"Intermediate, Advanced",Machine Learning Specialization
2,Computing & IT,Programming,"Beginner, Intermediate",AWS Cloud Technical Essentials
3,Business & Management,Quantitative Skills,"Beginner, Intermediate",Business Analytics for Decision Making
4,Psychology & Health Sciences,Research Methods,"Beginner, Intermediate",Understanding Research Methods


In [40]:
tie_test = recommend_resources(
    academic_area="Data Science & AI",

    confidence_scores={
        "Programming": 2,
        "Mathematics & Statistics": 2,
        "Data Analysis": 4
    },

    interests=[
        "Machine Learning"
    ],

    academic_goal="Improve academic performance",

    pass_probability=60.0
)

print("Priority skill:")
print(tie_test["priority_skill"])

print("Priority skills:")
print(tie_test["priority_skills"])

Priority skill:
Programming, Mathematics & Statistics
Priority skills:
['Programming', 'Mathematics & Statistics']


In [41]:
high_confidence_test = recommend_resources(
    academic_area="Data Science & AI",

    confidence_scores={
        "Programming": 4,
        "Mathematics & Statistics": 4,
        "Data Analysis": 4
    },

    interests=[
        "Machine Learning"
    ],

    academic_goal="Develop advanced technical skills",

    pass_probability=85.0
)

print("Priority skill:")
print(high_confidence_test["priority_skill"])

print("Priority skills:")
print(high_confidence_test["priority_skills"])

Priority skill:
No major low-confidence area identified
Priority skills:
[]


In [42]:
low_path_test = recommend_resources(
    academic_area="Data Science & AI",

    confidence_scores={
        "Programming": 2,
        "Mathematics & Statistics": 3,
        "Data Analysis": 4
    },

    interests=[
        "Machine Learning"
    ],

    academic_goal="Improve academic performance",

    pass_probability=35.0
)

print("Preferred difficulties:")
print(low_path_test["preferred_difficulties"])

low_path_test["learning_path"][
    [
        "Learning_Path_Stage",
        "Resource_Name",
        "Difficulty_Level",
        "Match_Score"
    ]
]

Preferred difficulties:
['Beginner']


,Learning_Path_Stage,Resource_Name,Difficulty_Level,Match_Score
15,1. Foundations,Python for Everybody,Beginner,48.8
14,1. Foundations,What is Data Science?,Beginner,27.9


In [43]:
moderate_path_test = recommend_resources(
    academic_area="Data Science & AI",

    confidence_scores={
        "Programming": 3,
        "Mathematics & Statistics": 3,
        "Data Analysis": 4
    },

    interests=[
        "Machine Learning"
    ],

    academic_goal="Improve academic performance",

    pass_probability=60.0
)

print("Preferred difficulties:")
print(moderate_path_test["preferred_difficulties"])

moderate_path_test["learning_path"][
    [
        "Learning_Path_Stage",
        "Resource_Name",
        "Difficulty_Level",
        "Match_Score"
    ]
]

Preferred difficulties:
['Beginner', 'Intermediate']


,Learning_Path_Stage,Resource_Name,Difficulty_Level,Match_Score
15,1. Foundations,Python for Everybody,Beginner,39.3
14,1. Foundations,What is Data Science?,Beginner,27.5
18,2. Core Skills,Machine Learning Specialization,Intermediate,64.3
16,2. Core Skills,Statistics for Data Science with Python,Intermediate,38.5


In [44]:
high_path_test = recommend_resources(
    academic_area="Data Science & AI",

    confidence_scores={
        "Programming": 4,
        "Mathematics & Statistics": 4,
        "Data Analysis": 4
    },

    interests=[
        "Machine Learning"
    ],

    academic_goal="Develop advanced technical skills",

    pass_probability=85.0
)

print("Preferred difficulties:")
print(high_path_test["preferred_difficulties"])

high_path_test["learning_path"][
    [
        "Learning_Path_Stage",
        "Resource_Name",
        "Difficulty_Level",
        "Match_Score"
    ]
]

Preferred difficulties:
['Intermediate', 'Advanced']


,Learning_Path_Stage,Resource_Name,Difficulty_Level,Match_Score
18,2. Core Skills,Machine Learning Specialization,Intermediate,55.3
16,2. Core Skills,Statistics for Data Science with Python,Intermediate,26.7
20,3. Advanced Topics,Data Science: Statistics and Machine Learning,Advanced,56.1
19,3. Advanced Topics,Deep Learning Specialization,Advanced,27.5


## 10. Preparation for Streamlit Integration

The recommendation logic is now consolidated into a reusable function so that the Streamlit prototype can generate personalised recommendations dynamically from student inputs.

The final recommendation component receives:
- academic area,
- confidence ratings,
- selected interests,
- academic goal,
- and predicted pass probability,

and returns:
- the priority skill,
- preferred difficulty levels,
- ranked learning resources,
- top recommendations,
- and a suggested learning pathway.

In [45]:
# Focused recommendation logic test summary

focused_test_summary = pd.DataFrame({
    "Test": [
        "Low Predicted Performance",
        "Moderate Predicted Performance",
        "High Predicted Performance",
        "Tied Low-Confidence Skills",
        "No Major Low-Confidence Area"
    ],

    "Expected_Behaviour": [
        "Beginner difficulty only",
        "Beginner and Intermediate difficulties",
        "Intermediate and Advanced difficulties",
        "Both lowest-confidence skills identified",
        "No priority skill identified"
    ],

    "Actual_Behaviour": [
        ", ".join(low_path_test["preferred_difficulties"]),
        ", ".join(moderate_path_test["preferred_difficulties"]),
        ", ".join(high_path_test["preferred_difficulties"]),
        ", ".join(tie_test["priority_skills"]),
        high_confidence_test["priority_skill"]
    ],

    "Result": [
        (
            "PASS"
            if low_path_test["preferred_difficulties"]
            == ["Beginner"]
            else "FAIL"
        ),

        (
            "PASS"
            if moderate_path_test["preferred_difficulties"]
            == ["Beginner", "Intermediate"]
            else "FAIL"
        ),

        (
            "PASS"
            if high_path_test["preferred_difficulties"]
            == ["Intermediate", "Advanced"]
            else "FAIL"
        ),

        (
            "PASS"
            if set(tie_test["priority_skills"])
            == {"Programming", "Mathematics & Statistics"}
            else "FAIL"
        ),

        (
            "PASS"
            if high_confidence_test["priority_skills"] == []
            else "FAIL"
        )
    ]
})

focused_test_summary

,Test,Expected_Behaviour,Actual_Behaviour,Result
0,Low Predicted Performance,Beginner difficulty only,Beginner,PASS
1,Moderate Predicted Performance,Beginner and Intermediate difficulties,"Beginner, Intermediate",PASS
2,High Predicted Performance,Intermediate and Advanced difficulties,"Intermediate, Advanced",PASS
3,Tied Low-Confidence Skills,Both lowest-confidence skills identified,"Programming, Mathematics & Statistics",PASS
4,No Major Low-Confidence Area,No priority skill identified,No major low-confidence area identified,PASS
